# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [6]:
import pandas as pd
import plotly.express as px
import numpy as np

# Load the dataset
df = pd.read_csv('global_energy_mix.csv')

# Categorize energy sources
source_category = {
    'Coal': 'Fossil',
    'Oil': 'Fossil',
    'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon',
    'Hydro': 'Low-carbon',
    'Wind': 'Renewable',
    'Solar': 'Renewable',
    'Other Renewables': 'Renewable'
}

# Add energy source categories
df['Source_Type'] = df['Source'].map(source_category)

# Display dataset information
print("Number of rows:", len(df))
print(df.head(10))


Number of rows: 103
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [7]:
fossil_data = df.loc[df['Source_Type'] == 'Fossil'].copy()

region_total = fossil_data.groupby('Region')['TWh'].sum()
country_total = fossil_data.groupby('Country')['TWh'].sum()

top_region = region_total.idxmax()
top_country = country_total.idxmax()

fig = px.treemap(
    fossil_data,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map={
        'Coal': '#0072B2',
        'Oil': '#E69F00',
        'Natural Gas': '#D55E00'
    },
    title=f'Fossil Fuel Dependency: {top_region} and {top_country}'
)

fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    marker=dict(
        colors=[
            {'Coal': '#0072B2',
             'Oil': '#E69F00',
             'Natural Gas': '#D55E00'}.get(label, '#BDBDBD')
            for label in fig.data[0].labels
        ]
    )
)

fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=50, b=10)
)

fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [8]:
low_carbon = df.loc[df['Source_Type'] == 'Low-carbon'].copy()

country_energy = low_carbon.groupby('Country', as_index=False)['TWh'].sum()

country_energy = country_energy.sort_values('TWh', ascending=False)

top_country = country_energy.iloc[0]['Country']

country_energy['All'] = 'Low-carbon'

fig = px.treemap(
    country_energy,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-Carbon Energy Production by Country'
)

fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh'
)

fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=50, b=10)
)

fig.show()


fig = px.bar(
    country_energy,
    x='TWh',
    y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=['#0072B2'],
    title=f'{top_country} Leads in Low-Carbon Energy Production'
)

fig.update_traces(
    texttemplate='%{x:,.0f} TWh',
    textposition='outside'
)

fig.update_layout(
    font=dict(family='Arial', size=12),
    xaxis_title='Energy Production (TWh)',
    yaxis_title='Country',
    yaxis=dict(categoryorder='total ascending'),
    margin=dict(l=10, r=40, t=50, b=10)
)

fig.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The horizontal bar chart shows the ranking more clearly because the countries are sorted by their total low-carbon energy production. It is easier to compare the values and identify the leading countries, such as France. The treemap is useful for showing the overall distribution of energy, but the bar chart makes comparisons between countries easier to understand.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

The treemap answers which countries and energy sources contribute the most to total fossil fuel consumption within each region. The 100% stacked bar chart answers which region depends most on coal compared to its total fossil fuel consumption. The treemap focuses on the absolute energy distribution, while the stacked bar chart focuses on the percentage share of each fuel source.

In [12]:
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

region_data = df_fossil.groupby(
    ['Region', 'Source'], as_index=False
)['TWh'].sum()

region_data['Total'] = region_data.groupby('Region')['TWh'].transform('sum')

region_data['Share'] = (region_data['TWh'] / region_data['Total']) * 100

coal_data = region_data[region_data['Source'] == 'Coal'].sort_values(
    'Share', ascending=False
)

top_region = coal_data.iloc[0]['Region']
coal_share = coal_data.iloc[0]['Share']

region_order = coal_data['Region'].tolist()

region_data['Label'] = region_data.apply(
    lambda row: f"{row['Share']:.1f}%" if row['Source'] == 'Coal' else '',
    axis=1
)

fig = px.bar(
    region_data,
    x='Share',
    y='Region',
    color='Source',
    orientation='h',
    text='Label',
    barmode='stack',
    category_orders={
        'Region': region_order,
        'Source': ['Coal', 'Oil', 'Natural Gas']
    },
    color_discrete_map={
        'Coal': '#0072B2',
        'Oil': '#BDBDBD',
        'Natural Gas': '#737373'
    },
    title=f'{top_region} Has the Highest Coal Share ({coal_share:.1f}%)'
)

fig.update_traces(
    textposition='inside',
    insidetextanchor='middle'
)

fig.update_layout(
    font=dict(family='Arial', size=12),
    xaxis=dict(
        title='Share of Fossil Energy (%)',
        range=[0, 100]
    ),
    yaxis=dict(
        title='Region',
        categoryorder='array',
        categoryarray=region_order
    ),
    margin=dict(l=10, r=10, t=55, b=10)
)

fig.show()